# 09 — Persistent Matmul

**Ladder:** Classic one program / tile (tutorial 03) → persistent schedule (few workers, many tiles) → Triton persistent kernel (`tl.num_programs` / work queue).

**Study tip.** Persistence changes **who runs which tile**, not $C=AB$. Learn static stride $p,p+P,\ldots$ first; then ask when an atomic work queue helps.

Official: [Persistent Matmul](https://triton-lang.org/main/getting-started/tutorials/09-persistent-matmul.html)

Primer: [`../03_matrix_multiplication/README.md`](../03_matrix_multiplication/README.md) · figures: [`docs/images/`](docs/images/)

![Whiteboard: classic one-tile programs vs persistent workers](docs/images/benchmark-persistent-matmul.png)

*Theory board — huge 1:1 tile grid vs few resident programs that steal tiles from a work queue.*

---

## 0. Two schedules for the same tiles

Let the output tile grid have $T = G_M G_N$ tiles with $G_M=\lceil M/B_M\rceil$, $G_N=\lceil N/B_N\rceil$. The **math** per tile is still the blocked product from §03:

$$
C_{\mathrm{tile}} = \sum_t A_t B_t
\quad\text{(FP32 accumulator, $K$-panels of width $B_K$)}.
$$

| Schedule | Grid size | Tile assignment |
|---|---|---|
| **Classic (naive launch)** | $T$ | program $p$ does tile $p$ once |
| **Persistent** | $P \ll T$ | program $p$ does tiles $p,\, p+P,\, p+2P,\ldots$ |
| **Work queue** | $P \ll T$ | `tile_id \leftarrow \mathrm{atomicAdd}(\mathrm{counter})$ (“tile stealing”) |

**Coverage lemma (stride).** For persistent launch with $P$ programs,

$$
\bigcup_{p=0}^{P-1} \{ p + kP \mid k \ge 0,\; p+kP < T \}
= \{0,\ldots,T-1\}.
$$

*Proof.* Every integer $t < T$ has unique $p = t \bmod P$ and $k = \lfloor t/P \rfloor$.

**Teaching sizes.** $M=64$, $N=48$, $B_M=32$, $B_N=16$ → $T=6$; try $P=3$ (two tiles each).

### Theory board

![Whiteboard: classic one-tile programs vs persistent workers](docs/images/whiteboard-theory.jpg)

*Theory board — huge 1:1 tile grid vs few resident programs that steal tiles from a work queue.*

### Pitfalls

- Persistent grid size ≈ #SMs (or small multiple), not #tiles.
- Need a correct work counter / tile id mapping.
- Over-persistence without enough tiles can hurt latency.

### Self-check

1. Contrast classic grid vs persistent worker loop.
2. Who decides the next tile — host or device counter?
3. When is persistence most helpful?


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import importlib.util

import pytest
import torch
import triton
import triton.language as tl

def _active_device():
    """Pick Triton active device, else CUDA, else CPU (study helper; no common/ package)."""
    try:
        import triton

        try:
            return triton.runtime.driver.active.get_active_torch_device()
        except Exception:
            pass
    except Exception:
        pass
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


DEVICE = _active_device()


def require_gpu() -> None:
    if not torch.cuda.is_available() and getattr(DEVICE, "type", None) != "cuda":
        pytest.skip(f"GPU not available (active device: {DEVICE})")


_matmul_path = Path(__file__).resolve().parents[1] / "03_matrix_multiplication" / "tutorial.py"
_spec = importlib.util.spec_from_file_location("matmul_mod", _matmul_path)
_matmul_mod = importlib.util.module_from_spec(_spec)
assert _spec.loader is not None
_spec.loader.exec_module(_matmul_mod)
matmul = _matmul_mod.matmul


## 1–2. Ladder: Classic → Persistent pseudo → Triton

**Corollary.** Any covering of the tile grid by programs that compute the correct block product yields the same $C$. Persistence changes **who** runs which tile and **when**, not the bilinear map $(A,B)\mapsto AB$.

| Stage | Implementation in this notebook |
|---|---|
| **Naive / classic** | `naive_blocked_matmul` → tutorial-03 `matmul` (one tile ↔ one short program in the underlying launch model) |
| **Intermediate** | `persistent_matmul_pseudo` — $P$ Python workers, stride schedule |
| **Triton** | `persistent_matmul_kernel` — `for tile_id in range(pid, num_tiles, num_programs)` |

**Index map** (row-major; optional grouped swizzle from 03):

$$
p_m = \big\lfloor \mathrm{tile\_id}/G_N \big\rfloor,
\qquad
p_n = \mathrm{tile\_id}\bmod G_N.
$$

In Triton: `pid = tl.program_id(0)`, `num_pid = tl.num_programs(0)`, then the loop above with the usual pointer / `tl.dot` / store body.

**When persistence helps.** Many small/medium tiles (launch / wave overhead); irregular tile costs (compose with Group GEMM, 08). Huge tiles with $t_{\mathrm{comp}}\gg L$ → little gain.


In [ ]:
def naive_blocked_matmul(A, B):
    """
    Classic blocked launch: grid ≈ (#tiles_M) * (#tiles_N).
    Each program owns exactly one output tile, then exits.

    We reuse tutorial 03's `matmul` (fused grouped GEMM) as that baseline.
    """
    return matmul(A, B)


In [ ]:
def persistent_matmul_pseudo(A, B, BLOCK_M, BLOCK_N, num_programs):
    """
    Persistent idea in Python:
    (See markdown cells above for the math.)
    """
    M, K = A.shape
    K2, N = B.shape
    assert K == K2
    assert M % BLOCK_M == 0 and N % BLOCK_N == 0

    C = torch.zeros(M, N, device=A.device, dtype=A.dtype)
    grid_m = M // BLOCK_M
    grid_n = N // BLOCK_N
    num_tiles = grid_m * grid_n

    for pid in range(num_programs):
        for tile_id in range(pid, num_tiles, num_programs):
            pid_m = tile_id // grid_n
            pid_n = tile_id % grid_n
            m0 = pid_m * BLOCK_M
            n0 = pid_n * BLOCK_N
            C[m0 : m0 + BLOCK_M, n0 : n0 + BLOCK_N] = (
                A[m0 : m0 + BLOCK_M, :] @ B[:, n0 : n0 + BLOCK_N]
            )
    return C


In [ ]:
@triton.jit
def persistent_matmul_kernel(
    A_ptr,
    B_ptr,
    C_ptr,
    M,
    N,
    K,
    stride_am,
    stride_ak,
    stride_bk,
    stride_bn,
    stride_cm,
    stride_cn,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):
    """Few programs; each strides across output tiles with tl.num_programs."""
    pid = tl.program_id(axis=0)
    num_pid = tl.num_programs(axis=0)

    num_pid_m = tl.cdiv(M, BLOCK_M)
    num_pid_n = tl.cdiv(N, BLOCK_N)
    num_tiles = num_pid_m * num_pid_n

    for tile_id in range(pid, num_tiles, num_pid):
        pid_m = tile_id // num_pid_n
        pid_n = tile_id % num_pid_n

        offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
        offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
        offs_k = tl.arange(0, BLOCK_K)

        a_ptrs = A_ptr + offs_m[:, None] * stride_am + offs_k[None, :] * stride_ak
        b_ptrs = B_ptr + offs_k[:, None] * stride_bk + offs_n[None, :] * stride_bn
        acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)

        for k in range(0, tl.cdiv(K, BLOCK_K)):
            k_remaining = K - k * BLOCK_K
            a = tl.load(
                a_ptrs,
                mask=(offs_m[:, None] < M) & (offs_k[None, :] < k_remaining),
                other=0.0,
            )
            b = tl.load(
                b_ptrs,
                mask=(offs_k[:, None] < k_remaining) & (offs_n[None, :] < N),
                other=0.0,
            )
            acc += tl.dot(a, b)
            a_ptrs += BLOCK_K * stride_ak
            b_ptrs += BLOCK_K * stride_bk

        c = acc.to(C_ptr.dtype.element_ty)
        c_ptrs = C_ptr + offs_m[:, None] * stride_cm + offs_n[None, :] * stride_cn
        tl.store(c_ptrs, c, mask=(offs_m[:, None] < M) & (offs_n[None, :] < N))


def persistent_matmul(a, b, num_programs=None, BLOCK_M=64, BLOCK_N=64, BLOCK_K=32):
    """Launch a small persistent grid; each program loops over many tiles."""
    M, K = a.shape
    K2, N = b.shape
    assert K == K2
    c = torch.empty((M, N), device=a.device, dtype=a.dtype)
    num_tiles = triton.cdiv(M, BLOCK_M) * triton.cdiv(N, BLOCK_N)
    if num_programs is None:
        num_programs = min(num_tiles, 8)
    persistent_matmul_kernel[(num_programs,)](
        a,
        b,
        c,
        M,
        N,
        K,
        a.stride(0),
        a.stride(1),
        b.stride(0),
        b.stride(1),
        c.stride(0),
        c.stride(1),
        BLOCK_M=BLOCK_M,
        BLOCK_N=BLOCK_N,
        BLOCK_K=BLOCK_K,
    )
    return c


In [ ]:
def test_pseudo_matches_torch() -> None:
    torch.manual_seed(0)
    a = torch.randn(64, 32)
    b = torch.randn(32, 48)
    ref = a @ b
    out = persistent_matmul_pseudo(a, b, BLOCK_M=32, BLOCK_N=16, num_programs=3)
    assert torch.allclose(out, ref, atol=1e-5)


def test_correctness() -> None:
    require_gpu()
    torch.manual_seed(0)
    a = torch.randn(256, 128, device=DEVICE, dtype=torch.float16)
    b = torch.randn(128, 256, device=DEVICE, dtype=torch.float16)
    ref = torch.matmul(a, b)

    baseline = naive_blocked_matmul(a, b)
    assert torch.allclose(baseline.float(), ref.float(), atol=1e-2, rtol=1e-2)

    out = persistent_matmul(a, b, num_programs=4, BLOCK_M=64, BLOCK_N=64, BLOCK_K=32)
    assert torch.allclose(out.float(), ref.float(), atol=1e-2, rtol=1e-2)
    print("persistent matmul OK")
